<div align="center">

<a href="https://colab.research.google.com/github/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-08/NB08_classification_evaluation.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 08: Classification and Evaluation: Defects, Failures and Grades

**Artificial Intelligence Applications in Engineering (MUH-920 Mühendislikte Yapay Zeka Uygulamaları)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)<br>[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. Last update: September 2026.</sub>

</div>

## How to use this notebook

This notebook is a self-contained version of the week: The lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. No earlier Python experience is assumed: Each new piece of Python appears in a cell marked **Python move**, which explains the syntax right where it is first needed. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: Open them only after a genuine attempt.

The animations of the lecture run on the [lecture page](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-08/lecture.html), and the interactive lab with the self-assessment is available at <https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-08/lab.html>. Cells that download public data need an internet connection. When a source is unreachable, the notebook falls back to a documented synthetic stand-in so that every later cell still runs.

## Overview

Many engineering decisions are categorical: accept or reject a part, which fault mode is developing, which variety a grain belongs to, whether a rock burst is likely. Classification learns such decisions from labelled examples. This week compares logistic regression, nearest neighbours, support vector machines and tree ensembles [1, 2], and spends as much time on evaluation as on models: confusion matrices, precision and recall, ROC and precision-recall curves, thresholds chosen by costs, and the treatment of rare failures [3, 4, 5]. The core cases are the dry bean dataset, produced with computer vision in Türkiye [6, 7], and the AI4I predictive maintenance dataset [8, 9].

**Estimated study time:** 9 to 11 hours.

## Learning outcomes

By the end of the week, students are expected to train and compare multiclass classifiers with scikit-learn pipelines and stratified splits, to compute and interpret a confusion matrix, precision, recall, F1 and their macro averages, to draw and read ROC and precision-recall curves, to choose a decision threshold from engineering costs, to handle class imbalance with class weights or resampling, and to explain a model's behaviour with permutation importance.

## Python in this week

The lecture ends with Python step 8: Grouping, counting and evaluation functions. It covers counting and grouping with pandas, boolean logic on columns, functions that return several values, and results collected in tables, applied to a machine failure log [10]. The hands-on part of the notebook opens with the same step as runnable cells and short exercises with immediate feedback, and its later sections mark further constructs as Python moves.

## Classification problems in engineering

A classifier assigns an input to one of several classes. Quality inspection separates good from defective parts, condition monitoring distinguishes fault modes, food engineering grades products, and geology assigns facies to depth intervals of a well. A classifier divides the input space into regions separated by decision boundaries. Most modern classifiers do more than name a class: They estimate a probability or a score for each class, and a decision rule, usually a threshold, turns the score into an action. Separating the model from the decision is essential in engineering, because the costs of different mistakes are rarely equal.

## Models for classification

Logistic regression models the log-odds of a class as a linear function of the inputs, so its decision boundary is a straight line or a plane, and its coefficients show the direction in which each input pushes the decision [11]. The k-nearest-neighbour classifier takes a vote among the most similar training examples [12]. Support vector machines find the boundary with the largest margin between classes and use kernels to draw curved boundaries in the original space [1]. Decision trees ask a sequence of threshold questions and are easy to read [13, 14]; random forests and gradient boosting combine many trees and are among the strongest methods for tabular engineering data [2, 15].


> **Animation: Decision boundaries of three classifiers.** The same two-class data are separated by logistic regression, k nearest neighbours and a decision tree. Change k and the depth and watch the boundary move between too rigid and too wiggly. Run it in the [web version of the lecture](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-08/lecture.html#anim-boundary).


![Confusion matrix of a random forest on the synthetic stand-in of the dry bean data. Most errors occur between classes with similar shapes.](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-08/figures/w08_fig1.png)

*Figure 8.1. Confusion matrix of a random forest on the synthetic stand-in of the dry bean data. Most errors occur between classes with similar shapes.*




<details>
<summary><b>Check your understanding.</b> Which classifier draws a linear decision boundary in the original input space?</summary>

A. Logistic regression without feature transformations  
B. k nearest neighbours with k = 1  
C. A random forest  
D. A support vector machine with a radial basis function kernel

**Answer: A.** Logistic regression models the log-odds linearly, so the boundary where the probability equals the threshold is a hyperplane.

</details>

## The confusion matrix and its metrics

For a binary problem with a positive class, such as "failure", the confusion matrix counts true positives (TP), false positives (FP), false negatives (FN) and true negatives (TN). Accuracy, the share of correct decisions, is misleading when classes are imbalanced: If 3 percent of machines fail, a model that always predicts "no failure" is 97 percent accurate and useless. Precision, TP / (TP + FP), is the share of alarms that are real. Recall, TP / (TP + FN), is the share of real failures that are caught. The F1 score is their harmonic mean. For several classes, the metrics are computed per class and averaged, either equally for every class (macro average) or weighted by class size (weighted average); the macro average reveals poor performance on small classes.


<details>
<summary><b>Check your understanding.</b> A model raises 50 alarms, of which 40 are real failures, and misses 10 failures. What are its precision and recall?</summary>

A. Precision 0.8 and recall 0.8  
B. Precision 0.8 and recall 0.4  
C. Precision 0.4 and recall 0.8  
D. Precision 1.0 and recall 0.8

**Answer: A.** Precision is 40 / 50 = 0.8 and recall is 40 / (40 + 10) = 0.8.

</details>

## Curves, thresholds and costs

Varying the threshold traces curves. The ROC curve plots the true positive rate against the false positive rate, and the area under it (AUC) equals the probability that a random positive example receives a higher score than a random negative one [3]. When positives are rare, the precision-recall curve is more informative, because the false positive rate can look small while alarms are still mostly false [4].

The threshold should come from the application. If a missed bearing failure costs twenty times more than an unnecessary inspection, the expected cost for each threshold on validation data can be computed, and the threshold with the lowest cost chosen. The default of 0.5 is only correct when both errors cost the same and the probabilities are well calibrated.


<details>
<summary><b>Check your understanding.</b> Positives are 1 percent of the data. Which curve is recommended to compare classifiers?</summary>

A. The ROC curve only  
B. The precision-recall curve  
C. The training loss curve  
D. The learning rate curve

**Answer: B.** With rare positives, precision reveals how many alarms are false, which the false positive rate hides [4].

</details>

## Imbalanced data

Rare events are the rule in engineering: failures, defects, accidents, rock bursts. Several remedies exist. Stratified splits keep the class proportions equal in training and test data. Class weights make errors on the rare class more expensive during training. Resampling duplicates rare examples or removes common ones, and SMOTE creates synthetic minority examples by interpolating between neighbouring minority samples [5]. None of these remedies creates information; the most effective step is often better features, more minority data or a problem definition that matches the decision.


![Precision-recall curves of three classifiers for rare machine failures on the synthetic stand-in of the AI4I data. The dashed line marks the share of failures, the precision of random guessing.](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-08/figures/w08_fig2.png)

*Figure 8.2. Precision-recall curves of three classifiers for rare machine failures on the synthetic stand-in of the AI4I data. The dashed line marks the share of failures, the precision of random guessing.*

## Explaining classifiers

Engineers need to know why a model decides. Permutation importance measures how much a performance metric drops when the values of one feature are shuffled, which breaks its link to the target. SHAP values distribute a single prediction among the features according to a game-theoretic rule and are widely used for tree ensembles [16]. In predictive maintenance, explanations connect model output to physical mechanisms such as heat dissipation or tool wear [8]. Newer methods study the geometry of a model's decision function, for example GEMEX, which the instructor developed as an open Python package [17]. Explanations are models of models and can mislead, so they should be checked against domain knowledge and, where possible, against deliberately constructed test cases.


<details>
<summary><b>Check your understanding.</b> Shuffling the values of &quot;torque&quot; in the test data lowers the F1 score of a failure classifier from 0.82 to 0.41. What does this indicate?</summary>

A. Torque is unimportant for the model  
B. The model relies strongly on torque for its decisions  
C. Torque causes failures in every case  
D. The test data are leaked

**Answer: B.** A large drop after permutation shows that the model uses the feature; it does not prove a causal mechanism.

</details>

# Hands-on lab

The notebook classifies seven dry bean varieties from sixteen shape features [7] with logistic regression, k-nearest neighbours, a support vector machine, a random forest and gradient boosting, and studies the confusion matrix and permutation importance of the best model. It then turns to rare machine failures in the AI4I data [9]: the accuracy paradox, class weights, ROC and precision-recall curves, and a threshold chosen from the costs of misses and false alarms. A switch reruns the multiclass comparison on the dataset of another department.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report, f1_score, accuracy_score, roc_curve, precision_recall_curve,
                             roc_auc_score, average_precision_score)
from sklearn.inspection import permutation_importance

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] {'Correct.' if ok else 'Not yet. Check the logic and run again.'}")

## Python step 8: Grouping, counting and evaluation functions

This part follows the Python step at the end of the lecture. Run the cells in order, compare each result with the explanation, and change values to see what happens. The exercises at the end of the step give immediate feedback.

### Counting and grouping

Classification data raise two questions at once: How many examples does each class have, and how do the classes differ? `value_counts` counts the occurrences of each value in a column. `groupby` splits a table by the values of one column, applies a summary to each group and combines the results. The practice log below lists invented machine records with the product type, the tool wear, the torque and whether a failure occurred.

In [ ]:
import numpy as np
import pandas as pd

log = pd.DataFrame({
    "type": ["L", "M", "H", "L", "L", "M", "H", "L", "M", "L", "H", "M"],
    "tool_wear_min": [12, 180, 205, 221, 64, 230, 15, 199, 98, 240, 150, 30],
    "torque_Nm": [38.2, 45.1, 61.0, 52.3, 40.0, 58.4, 33.5, 49.9, 41.2, 63.8, 60.5, 36.1],
    "failure": [0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0],
})
print(log["type"].value_counts())
print(log.groupby("type")["failure"].mean().round(2))
print(log.groupby("failure")[["tool_wear_min", "torque_Nm"]].mean().round(1))

### A rule as a classifier

A simple rule predicts a failure when the tool wear exceeds 200 minutes or the torque exceeds 60 N m. The combined condition produces a boolean Series, and `astype(int)` turns it into zeros and ones. `pd.crosstab` counts the combinations of actual and predicted classes, which is the confusion matrix of this week.

In [ ]:
pred = ((log["tool_wear_min"] > 200) | (log["torque_Nm"] > 60)).astype(int)
actual = log["failure"]
print(pd.crosstab(actual, pred, rownames=["actual"], colnames=["predicted"]))

### Functions that return several values

A function can return several values separated by commas. Python packs them into a tuple, and the caller unpacks them into separate names. The function below counts true positives, false positives, false negatives and true negatives. A loop over several wear limits collects one dictionary per limit in a list, and `pd.DataFrame` turns the list into a table of results, a pattern that also serves to compare models, settings or datasets.

In [ ]:
def confusion(y_true, y_pred):
    """Return the counts (tp, fp, fn, tn) for labels coded as 0 and 1."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    return tp, fp, fn, tn


tp, fp, fn, tn = confusion(actual, pred)
print("tp, fp, fn, tn =", (tp, fp, fn, tn))
rows = []
for limit in [150, 200, 220]:
    tp, fp, fn, tn = confusion(actual, log["tool_wear_min"] > limit)
    rows.append({"wear limit": limit, "precision": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1)})
print(pd.DataFrame(rows).round(2))

The expression `max(tp + fp, 1)` avoids a division by zero when a rule predicts no failure at all.

**Quick check.** A function ends with return tp, fp, fn, tn. Which line stores its four results under separate names?

A. tp, fp, fn, tn = confusion(y, p)  
B. tp = fp = fn = tn = confusion(y, p)  
C. [tp] = confusion(y, p)  
D. confusion(y, p).unpack()

<details><summary>Show the answer</summary>

**A.** The function returns a tuple of four values, and a tuple unpacks into as many names as it has elements.

</details>

### Exercises for Python step 8

**Exercise 8.1.** Compute the accuracy of the rule `pred`, the share of rows where the prediction equals the actual class, and store it in `py_accuracy`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py8_log():
    import pandas as pd
    return pd.DataFrame({"type": ["L", "M", "H", "L", "L", "M", "H", "L", "M", "L", "H", "M"],
                         "tool_wear_min": [12, 180, 205, 221, 64, 230, 15, 199, 98, 240, 150, 30],
                         "torque_Nm": [38.2, 45.1, 61.0, 52.3, 40.0, 58.4, 33.5, 49.9, 41.2, 63.8, 60.5, 36.1],
                         "failure": [0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0]})


def _py8_accuracy_reference():
    lg = _py8_log()
    p = ((lg["tool_wear_min"] > 200) | (lg["torque_Nm"] > 60)).astype(int)
    return float((p == lg["failure"]).mean())

In [ ]:
py_accuracy = None
check("Exercise 8.1", py_accuracy, _py8_accuracy_reference())

**Exercise 8.2.** Use `groupby` to find the failure rate of product type L and store it in `py_rate_L`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py8_rate_L_reference():
    lg = _py8_log()
    return float(lg.groupby("type")["failure"].mean()["L"])

In [ ]:
py_rate_L = None
check("Exercise 8.2", py_rate_L, _py8_rate_L_reference())

**Exercise 8.3.** Use `confusion` to compute the recall of the rule "tool wear above 150 minutes" and store it in `py_recall_150`. What happens to the precision at the same time?

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py8_recall_150_reference():
    lg = _py8_log()
    p = lg["tool_wear_min"] > 150
    y = lg["failure"] == 1
    return float((p & y).sum() / y.sum())

In [ ]:
py_recall_150 = None
check("Exercise 8.3", py_recall_150, _py8_recall_150_reference())

### A reusable loader for public datasets

**Python move: a reusable function with keyword arguments.** `load_uci` downloads a dataset from the UCI Machine Learning Repository with the `ucimlrepo` package [18] and returns the features as a pandas DataFrame and the target as a Series. The package is installed on first use. If the repository cannot be reached, the calling cell switches to a documented synthetic stand-in, so that every later cell still runs.

In [ ]:
import subprocess
import sys


def load_uci(uci_id, target=None, drop=(), onehot_targets=False):
    """Return (X, y) for a UCI dataset; target names a column of the targets or of the features."""
    try:
        from ucimlrepo import fetch_ucirepo
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--retries", "1", "--timeout", "15", "ucimlrepo"], check=False)
        from ucimlrepo import fetch_ucirepo
    data = fetch_ucirepo(id=uci_id).data
    X = data.features.copy()
    Y = data.targets.copy() if data.targets is not None else pd.DataFrame(index=X.index)
    if onehot_targets:                                   # several 0/1 target columns, one per class
        y = Y.idxmax(axis=1)
    else:
        if target is None:
            target = Y.columns[0]
        y = X.pop(target) if target in X.columns else Y[target]
    X = X.drop(columns=[c for c in drop if c in X.columns])
    return X, y.astype(float) if y.dtype.kind in "iuf" else y

## 1. Dry beans from computer vision

Koklu and Ozkan photographed 13,611 grains of seven registered dry bean varieties and computed twelve dimension features and four shape factors from the images [6, 7]. The synthetic stand-in, used only when the download fails, builds beans as noisy ellipses with class-specific sizes and computes the same sixteen features from them.

In [ ]:
def bean_stand_in(seed=602):
    rng = np.random.default_rng(seed)
    spec = {"DERMASON": (3546, 240, 170), "SIRA": (2636, 310, 210), "SEKER": (2027, 255, 200), "HOROZ": (1928, 370, 185),
            "CALI": (1630, 410, 245), "BARBUNYA": (1322, 370, 240), "BOMBAY": (522, 590, 380)}
    rows, labels = [], []
    for name, (n, L0, l0) in spec.items():
        L = rng.normal(L0, 0.07 * L0, n); l = np.minimum(rng.normal(l0, 0.07 * l0, n), 0.98 * L)
        area = np.pi * L * l / 4 * rng.normal(0.99, 0.01, n)
        perim = np.pi * (3 * (L + l) / 4 - np.sqrt((3 * L + l) * (L + 3 * l)) / 4) * rng.normal(1.01, 0.01, n)
        convex = area * rng.normal(1.01, 0.004, n)
        eqd = np.sqrt(4 * area / np.pi)
        rows.append(pd.DataFrame({"Area": area, "Perimeter": perim, "MajorAxisLength": L, "MinorAxisLength": l, "AspectRation": L / l,
                                  "Eccentricity": np.sqrt(1 - (l / L) ** 2), "ConvexArea": convex, "EquivDiameter": eqd,
                                  "Extent": area / (L * l) * rng.normal(1, 0.02, n), "Solidity": area / convex, "roundness": 4 * np.pi * area / perim**2,
                                  "Compactness": eqd / L, "ShapeFactor1": L / area, "ShapeFactor2": l / area,
                                  "ShapeFactor3": area / (np.pi * (L / 2) ** 2), "ShapeFactor4": area / (np.pi * L * l / 4)}))
        labels += [name] * n
    return pd.concat(rows, ignore_index=True), pd.Series(labels, name="Class")


try:
    Xb, yb = load_uci(602)
    BEAN_SOURCE = "UCI 602, Dry Bean"
except Exception as error:
    print("Download failed, using the synthetic stand-in:", type(error).__name__)
    Xb, yb = bean_stand_in()
    BEAN_SOURCE = "synthetic stand-in for UCI 602"
print(BEAN_SOURCE, Xb.shape)
yb.value_counts()

**Python move: stratified split.** `stratify=yb` keeps the class proportions equal in the training and test sets, which matters when classes differ in size.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.25, stratify=yb, random_state=0)
MODELS = {
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "k-NN (k 9)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=9)),
    "SVM (RBF)": make_pipeline(StandardScaler(), SVC(C=10, gamma="scale")),
    "random forest": RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1),
    "gradient boosting": HistGradientBoostingClassifier(random_state=0),
}
scores = {}
for name, model in MODELS.items():
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    scores[name] = {"accuracy": accuracy_score(yte, pred), "macro F1": f1_score(yte, pred, average="macro")}
pd.DataFrame(scores).T.round(4)

## 2. The confusion matrix of the best model

In [ ]:
best_name = max(scores, key=lambda k: scores[k]["macro F1"])
best = MODELS[best_name]
fig, ax = plt.subplots(figsize=(6.2, 5.2))
ConfusionMatrixDisplay.from_estimator(best, Xte, yte, ax=ax, xticks_rotation=45, colorbar=False, cmap="Blues")
ax.set_title(best_name); plt.tight_layout(); plt.show()
print(classification_report(yte, best.predict(Xte), digits=3))

### Your turn, exercise 1

A classifier for a two-class inspection problem produced TP = 45, FP = 15, FN = 5 and TN = 935. Compute the F1 score from precision and recall and store it in `f1_manual`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _f1_manual_reference():
    p, r = 45 / (45 + 15), 45 / (45 + 5)
    return 2 * p * r / (p + r)

In [ ]:
f1_manual = None   # your computation
check("Exercise 1", f1_manual, _f1_manual_reference(), tol=1e-6)

## 3. Which features does the model use?

Permutation importance shuffles one feature at a time in the test data and records the drop of the macro F1 score. A subsample keeps the computation quick.

In [ ]:
sample = Xte.sample(min(1500, len(Xte)), random_state=0)
imp = permutation_importance(best, sample, yte.loc[sample.index], scoring="f1_macro", n_repeats=5, random_state=0, n_jobs=-1)
order = np.argsort(imp.importances_mean)[::-1][:10]
plt.figure(figsize=(6, 3.6)); plt.barh(sample.columns[order][::-1], imp.importances_mean[order][::-1])
plt.xlabel("drop in macro F1 when shuffled"); plt.title("Permutation importance"); plt.tight_layout(); plt.show()

### Your turn, exercise 2

Train a random forest with 200 trees and `random_state=0` on the bean training data and store its macro F1 on the test set in `rf_macro_f1`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _rf_macro_f1_reference():
    rf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr, ytr)
    return float(f1_score(yte, rf.predict(Xte), average="macro"))

In [ ]:
rf_macro_f1 = None   # your computation
check("Exercise 2", rf_macro_f1, _rf_macro_f1_reference(), tol=1e-6)

## 4. Rare failures: the AI4I predictive maintenance data

The AI4I 2020 dataset models a milling process with 10,000 records and failures caused by tool wear, heat dissipation, power and overstrain, plus random failures [8, 9]. The failure-mode columns are removed from the inputs, because they are labels themselves. The stand-in follows the failure mechanisms documented for the dataset.

In [ ]:
def ai4i_stand_in(n=10000, seed=601):
    rng = np.random.default_rng(seed)
    typ = rng.choice(["L", "M", "H"], n, p=[0.5, 0.3, 0.2])
    air = 300 + 2 * np.sin(np.arange(n) / 800) + rng.normal(0, 0.5, n)
    proc = air + 10 + rng.normal(0, 0.8, n)
    torque = np.clip(rng.normal(40, 9, n), 3.8, 76.6)
    rpm = np.clip(1540 + (40 - torque) * 25 + rng.normal(0, 120, n), 1160, 2890)
    wear = rng.uniform(0, 230, n)
    power = torque * rpm * 2 * np.pi / 60
    limit = np.select([typ == "L", typ == "M"], [11000, 12000], 13000)
    fail = (((wear >= 200) & (rng.random(n) < 0.05)) | ((proc - air < 8.6) & (rpm < 1380)) | (power < 3500) | (power > 9000)
            | (wear * torque > limit) | (rng.random(n) < 0.001))
    X = pd.DataFrame({"Type": typ, "Air temperature": air, "Process temperature": proc, "Rotational speed": rpm, "Torque": torque, "Tool wear": wear})
    return X, pd.Series(fail.astype(int), name="Machine failure")


try:
    Xm, ym = load_uci(601, target="Machine failure", drop=("UDI", "Product ID", "TWF", "HDF", "PWF", "OSF", "RNF"))
    PM_SOURCE = "UCI 601, AI4I 2020"
except Exception as error:
    print("Download failed, using the synthetic stand-in:", type(error).__name__)
    Xm, ym = ai4i_stand_in()
    PM_SOURCE = "synthetic stand-in for UCI 601"
Xm = pd.get_dummies(Xm, columns=list(Xm.select_dtypes(exclude="number").columns), drop_first=True).astype(float)
print(PM_SOURCE, Xm.shape, "failure share:", round(float(ym.mean()), 4))

### The accuracy paradox, class weights and curves

In [ ]:
Xa, Xv, ya, yv = train_test_split(Xm, ym, test_size=0.3, stratify=ym, random_state=1)
pm_models = {
    "always 'no failure'": DummyClassifier(strategy="most_frequent"),
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "logistic, class weights": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
    "gradient boosting": HistGradientBoostingClassifier(random_state=0),
}
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for name, model in pm_models.items():
    model.fit(Xa, ya)
    p = model.predict_proba(Xv)[:, 1]
    print(f"{name:26s} accuracy {accuracy_score(yv, p >= 0.5):.3f}  recall {np.mean(p[yv == 1] >= 0.5):.3f}  "
          f"ROC AUC {roc_auc_score(yv, p):.3f}  AP {average_precision_score(yv, p):.3f}")
    if name != "always 'no failure'":
        fpr, tpr, _ = roc_curve(yv, p); pr, rc, _ = precision_recall_curve(yv, p)
        ax[0].plot(fpr, tpr, label=name); ax[1].plot(rc, pr, label=name)
ax[0].plot([0, 1], [0, 1], "k--", lw=0.8); ax[0].set_xlabel("false positive rate"); ax[0].set_ylabel("true positive rate"); ax[0].set_title("ROC")
ax[1].axhline(ym.mean(), color="k", ls="--", lw=0.8); ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].set_title("Precision-recall")
ax[1].legend(fontsize=7); plt.tight_layout(); plt.show()

## 5. A threshold from costs

A missed failure is assumed to cost 20 units and an unnecessary inspection 1 unit. The expected cost is computed on the validation data for thresholds from 0.01 to 0.99.

### Your turn, exercise 3

Using the gradient boosting probabilities `p_gb` below, find the threshold on the grid `np.round(np.arange(0.01, 1.0, 0.01), 2)` with the lowest total cost 20 FN + 1 FP and store it in `best_threshold`. If several thresholds tie, take the smallest.

In [ ]:
p_gb = pm_models["gradient boosting"].predict_proba(Xv)[:, 1]
y_v = yv.to_numpy()
grid = np.round(np.arange(0.01, 1.0, 0.01), 2)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _best_threshold_reference():
    cost = [20 * np.sum((p_gb < t) & (y_v == 1)) + np.sum((p_gb >= t) & (y_v == 0)) for t in grid]
    return float(grid[int(np.argmin(cost))])

In [ ]:
best_threshold = None   # your computation
check("Exercise 3", best_threshold, _best_threshold_reference(), tol=1e-9)

## 6. The discipline switch

Set `CHOICE` and run the cell to compare three classifiers with stratified five-fold cross-validation and the macro F1 on another dataset [19, 20, 21, 22, 23, 24, 25].

In [ ]:
SWITCH = {
    "rice varieties (food, biology)": dict(uci_id=545),
    "steel plate faults (mechanical, textile)": dict(uci_id=198, onehot_targets=True),
    "seismic bumps (mining, geophysics)": dict(uci_id=266),
    "MAGIC gamma telescope (physics)": dict(uci_id=159),
    "QSAR biodegradation (chemistry, environmental)": dict(uci_id=254),
    "smart grid stability (electrical)": dict(uci_id=471, target="stabf", drop=("stab",)),
    "wine quality 7 or more (food, chemical)": dict(uci_id=186, target="quality"),
}
CHOICE = "steel plate faults (mechanical, textile)"
try:
    Xd, yd = load_uci(**SWITCH[CHOICE])
    if CHOICE.startswith("wine"):
        yd = (yd >= 7).astype(int)
    Xd = pd.get_dummies(Xd, drop_first=True).astype(float)
    Xd = Xd.fillna(Xd.median())
except Exception as error:
    print(f"Could not load {CHOICE} ({type(error).__name__}); using the bean data instead.")
    Xd, yd = Xb, yb
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for name in ["logistic regression", "random forest", "gradient boosting"]:
    s = cross_val_score(MODELS[name], Xd, yd, cv=cv, scoring="f1_macro", n_jobs=-1)
    print(f"{name:20s} macro F1 {s.mean():.3f} +/- {s.std():.3f}")

## Discipline challenges

The switch in the notebook loads a classification dataset for several fields. Pick the row of your department and repeat the evaluation, paying attention to class balance and error costs.

| Department | Challenge |
|---|---|
| Food Engineering and Biology | Dry bean varieties from shape features, UCI 602 [6], or rice varieties, UCI 545 [26]. |
| Mechanical Engineering and Industrial Engineering | Machine failure in the AI4I 2020 predictive maintenance data, UCI 601 [9]. |
| Mechanical Engineering and Textile Engineering | Fault types of steel plates from geometric and luminosity features, UCI 198 [20]. |
| Mining Engineering and Geophysical Engineering | Hazardous seismic bumps in a coal mine, UCI 266 [21, 27]. |
| Physics | Gamma-ray showers versus hadron background in the MAGIC telescope, UCI 159 [22]. |
| Chemistry and Environmental Engineering | Ready biodegradability of chemicals from molecular descriptors, UCI 254 [23]. |
| Electrical and Electronics Engineering | Stability of a simulated decentralised smart grid, UCI 471, without the leaking stability margin [24]. |
| Chemical Engineering and Food Engineering | Good versus ordinary wine from physicochemical tests, UCI 186 [28]. |
| Geological Engineering and Earth Sciences Engineering | Lithofacies from well logs, the SEG facies data of Week 9 [29]. |
| Computer Engineering and Statistics | Compare macro and weighted averages and calibration on any multiclass dataset; discuss when each is appropriate. |

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-08/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Which classification decision in your field has unequal error costs? Estimate the ratio of the costs of a miss and a false alarm.
2. Which class of the dry bean data was hardest to recognise, and what does the confusion matrix suggest about the reason?
3. How would you explain the difference between precision and recall to a colleague in production, without formulas?

## Weekly task and submission

Choose a classification dataset from the discipline switch or from DATASETS.md, train at least three classifiers with a stratified split, and report the confusion matrix, per-class precision and recall, the macro F1 and a ROC or precision-recall curve. If the classes are imbalanced, choose a threshold from explicitly stated costs and justify them. Explain the most important features with permutation importance and discuss whether they make physical sense, in about 500 words, citing at least three works from this week's references [3, 4, 8].

The weekly task is optional and supports self-learning and a personal portfolio. During an active semester, it can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com. Optional work carries no separate weight, but the instructor may take it into account as a discretionary adjustment of the midterm component of the grade.

## Research and report assignment (optional)

**Evaluating classifiers for rare engineering events.** Review how studies on predictive maintenance, defect detection or hazard prediction evaluate classifiers when events are rare. Compare the metrics, resampling strategies and threshold choices, and assess whether the reported results support the claimed industrial value [4, 5, 8, 27].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com. It carries no separate weight, but the instructor may take it into account as a discretionary adjustment of the midterm component of the grade. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and uses the template in `exams/REPORT_TEMPLATE.md`.

## References

[1] Cortes, C., & Vapnik, V. (1995). Support-vector networks. *Machine Learning*, *20*(3), 273-297. <https://doi.org/10.1007/BF00994018>

[2] Breiman, L. (2001). Random forests. *Machine Learning*, *45*(1), 5-32. <https://doi.org/10.1023/A:1010933404324>

[3] Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters*, *27*(8), 861-874. <https://doi.org/10.1016/j.patrec.2005.10.010>

[4] Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE*, *10*(3), e0118432. <https://doi.org/10.1371/journal.pone.0118432>

[5] Chawla, N. V., Bowyer, K. W., Hall, L. O., & Kegelmeyer, W. P. (2002). SMOTE: Synthetic minority over-sampling technique. *Journal of Artificial Intelligence Research*, *16*, 321-357. <https://doi.org/10.1613/jair.953>

[6] Koklu, M., & Ozkan, I. A. (2020). Multiclass classification of dry beans using computer vision and machine learning techniques. *Computers and Electronics in Agriculture*, *174*, 105507. <https://doi.org/10.1016/j.compag.2020.105507>

[7] Koklu, M., & Ozkan, I. A. (2020). *Dry Bean [Dataset]. UCI Machine Learning Repository, ID 602*. <https://doi.org/10.24432/C50S4B>

[8] Matzka, S. (2020). Explainable artificial intelligence for predictive maintenance applications. In *2020 Third International Conference on Artificial Intelligence for Industries (AI4I)* (pp. 69-74). IEEE. <https://doi.org/10.1109/AI4I49448.2020.00023>

[9] Matzka, S. (2020). *AI4I 2020 Predictive Maintenance Dataset [Dataset]. UCI Machine Learning Repository, ID 601*. <https://doi.org/10.24432/C5HS5C>

[10] McKinney, W. (2010). Data structures for statistical computing in Python. In *Proceedings of the 9th Python in Science Conference* (pp. 56-61). <https://doi.org/10.25080/Majora-92bf1922-00a>

[11] Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning: Data Mining, Inference, and Prediction* (2nd ed.). Springer. <https://doi.org/10.1007/978-0-387-84858-7>

[12] Cover, T., & Hart, P. (1967). Nearest neighbor pattern classification. *IEEE Transactions on Information Theory*, *13*(1), 21-27. <https://doi.org/10.1109/TIT.1967.1053964>

[13] Breiman, L., Friedman, J. H., Olshen, R. A., & Stone, C. J. (1984). *Classification and Regression Trees*. Wadsworth.

[14] Quinlan, J. R. (1986). Induction of decision trees. *Machine Learning*, *1*(1), 81-106. <https://doi.org/10.1007/BF00116251>

[15] Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. In *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining* (pp. 785-794). <https://doi.org/10.1145/2939672.2939785>

[16] Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. In *Advances in Neural Information Processing Systems 30* (pp. 4765-4774). <https://arxiv.org/abs/1705.07874>

[17] Kose, U. (2026). *GEMEX: Geodesic Entropic Manifold Explainability (Version 1.2.2) [Python package]*. Python Package Index. <https://pypi.org/project/gemex/>

[18] Kelly, M., Longjohn, R., & Nottingham, K. (2026). *The UCI Machine Learning Repository*. <https://archive.ics.uci.edu>

[19] Cinar, I., & Koklu, M. (2019). *Rice (Cammeo and Osmancik) [Dataset]. UCI Machine Learning Repository, ID 545*. <https://doi.org/10.24432/C5MW4Z>

[20] Buscema, M., Terzi, S., & Tastle, W. (2010). *Steel Plates Faults [Dataset]. UCI Machine Learning Repository, ID 198*. <https://doi.org/10.24432/C5J88N>

[21] Sikora, M., & Wróbel, Ł. (2010). *seismic-bumps [Dataset]. UCI Machine Learning Repository, ID 266*. <https://doi.org/10.24432/C5W902>

[22] Bock, R. (2004). *MAGIC Gamma Telescope [Dataset]. UCI Machine Learning Repository, ID 159*. <https://doi.org/10.24432/C52C8B>

[23] Mansouri, K., Ringsted, T., Ballabio, D., Todeschini, R., & Consonni, V. (2013). *QSAR biodegradation [Dataset]. UCI Machine Learning Repository, ID 254*. <https://doi.org/10.24432/C5H60M>

[24] Arzamasov, V. (2018). *Electrical Grid Stability Simulated Data [Dataset]. UCI Machine Learning Repository, ID 471*. <https://doi.org/10.24432/C5PG66>

[25] Cortez, P., Cerdeira, A., Almeida, F., Matos, T., & Reis, J. (2009). *Wine Quality [Dataset]. UCI Machine Learning Repository, ID 186*. <https://doi.org/10.24432/C56S3T>

[26] Cinar, I., & Koklu, M. (2019). Classification of rice varieties using artificial intelligence methods. *International Journal of Intelligent Systems and Applications in Engineering*, *7*(3), 188-194. <https://doi.org/10.18201/ijisae.2019355381>

[27] Sikora, M., & Wróbel, Ł. (2010). Application of rule induction algorithms for analysis of data collected by seismic hazard monitoring systems in coal mines. *Archives of Mining Sciences*, *55*(1), 91-114.

[28] Cortez, P., Cerdeira, A., Almeida, F., Matos, T., & Reis, J. (2009). Modeling wine preferences by data mining from physicochemical properties. *Decision Support Systems*, *47*(4), 547-553. <https://doi.org/10.1016/j.dss.2009.05.016>

[29] Hall, B. (2016). Facies classification using machine learning. *The Leading Edge*, *35*(10), 906-909. <https://doi.org/10.1190/tle35100906.1>